# Day 4 — Solution: The Gauss–Markov Assumptions

In [ ]:
import os, sys, pathlib
root = pathlib.Path.cwd()
for _ in range(6):
    if (root / "qrc").is_dir():
        break
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from scipy import stats
DATA_SOURCE = os.environ.get("QRC_DATA", "real")

def ols_fit(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    X = np.column_stack([np.ones(n), x])
    XtX_inv = np.linalg.inv(X.T @ X)
    b = XtX_inv @ X.T @ y
    e = y - X @ b
    s2 = e @ e / (n - X.shape[1])
    se = np.sqrt(np.diag(s2 * XtX_inv))
    r2 = 1 - (e @ e) / ((y - y.mean()) @ (y - y.mean()))
    return {"alpha": b[0], "beta": b[1], "se_beta": se[1], "t_beta": b[1]/se[1], "r2": r2, "resid": e}

## E1 — A1–A6 from memory

Exemplar (compare against yours — agreement on *consequence* matters more
than wording):

- **A1 linearity**: y = Xβ + ε is the form — *buys: the model is the model.*
- **A2 exogeneity**: E[ε|X] = 0 — *buys: unbiasedness of β̂.*
- **A3 no perfect collinearity**: X full rank — *buys: (XᵀX)⁻¹ exists.*
- **A4 homoskedasticity**: Var(ε_t) = σ² — *buys: valid classical SEs, OLS efficiency.*
- **A5 no autocorrelation**: Cov(ε_t, ε_s) = 0 — *buys: valid classical SEs in time series.*
- **A6 normality** (optional): ε ~ N — *buys: exact small-sample t/F.*

**Note.** The usual memory hole: A4/A5 are NOT what makes β̂ unbiased, and
A6 is not an unbiasedness assumption at all. Also commonly dropped: the
intercept discipline from days 1–2 lives inside A1 (a missing constant is a
linear-form error).

## E2 — the tournament

In [ ]:
def tournament(world, n=1200, n_runs=600, beta=0.9, alpha=0.0001):
    est, se, rej = [], [], 0
    for i in range(n_runs):
        rng = np.random.default_rng(9000 + i)
        if world == "a_gm":
            m = rng.normal(0, 0.011, n)
            e = rng.normal(0, 0.010, n)
            y = alpha + beta * m + e
        elif world == "b_het":
            m = rng.normal(0, 0.011, n)
            sig = 0.006 * (0.4 + 3.5 * np.abs(m) / 0.011)      # vol scales with |m|
            e = rng.normal(0, 1, n) * sig
            y = alpha + beta * m + e
        elif world == "c_ac":
            m = np.zeros(n); u = rng.normal(0, 0.011, n)
            for t in range(1, n): m[t] = 0.95 * m[t-1] + u[t]  # persistent regressor
            e = np.zeros(n); v = rng.normal(0, 0.010, n)
            for t in range(1, n): e[t] = 0.7 * e[t-1] + v[t]   # persistent error
            y = alpha + beta * m + e
        elif world == "d_omit":
            m = rng.normal(0, 0.011, n)
            h = 0.7 * m + np.sqrt(1 - 0.7**2) * rng.normal(0, 0.011, n)
            y = alpha + beta * m + 0.5 * h + rng.normal(0, 0.010, n)
            # and we regress y on m ALONE:
        f = ols_fit(m, y)
        est.append(f["beta"]); se.append(f["se_beta"])
        rej += abs((f["beta"] - beta) / f["se_beta"]) > 1.96
    return np.mean(est), np.std(est), np.mean(se), rej / n_runs

rows = []
for w in ["a_gm", "b_het", "c_ac", "d_omit"]:
    rows.append([w] + list(tournament(w)))
tab = pd.DataFrame(rows, columns=["world", "E[beta] (0.90)", "sd(beta)", "mean SE (classical)", "rej rate (5% nominal)"])
print(tab.round(3).to_string(index=False))

**Expected pattern.** (a) GM: E[β̂] = 0.90, sd ≈ mean SE, rejection ≈ 5% —
the machinery as designed. (b) heteroskedastic (variance scaling with |m|):
β̂ still unbiased, but the sampling sd **exceeds** the mean classical SE
and rejection rises well above 5% — the classical formula averages the
variance over time while the information content concentrates in big-|m|
observations. Day 5 repairs exactly this. (c) persistent regressor ×
persistent errors: the true sd of β̂ runs ~2.2× the mean classical SE,
rejection ~35% — autocorrelated errors with a *persistent regressor* is
the dangerous pairing (day 8; note the same AR error with an iid
regressor barely dents anything — work out why in the margin).
(d) omitted variable: E[β̂] ≈ 0.9 + 0.5·0.7 ≈ 1.25 — **bias, the only
failure of unbiasedness in the table, and the one no sandwich estimator
repairs.** The four-by-four is the week's argument in one screen: A4/A5
failures corrupt inference, A2 failure corrupts substance.

## E3 — omitted-variable bias, by hand

In [ ]:
# plim beta_hat = beta + gamma * cov(m, h)/var(m) ; here gamma=0.5, cov/var = 0.7 (equal vols, corr 0.7)
ovb = 0.9 + 0.5 * 0.7
print(f"prediction: {ovb:.3f}")
# direct check on one long run
rng = np.random.default_rng(1)
n = 200_000
m = rng.normal(0, 0.011, n)
h = 0.7 * m + np.sqrt(1 - 0.7**2) * rng.normal(0, 0.011, n)
y = 0.0001 + 0.9 * m + 0.5 * h + rng.normal(0, 0.010, n)
print(f"one long run:  {ols_fit(m, y)['beta']:.3f}")

**Expected pattern.** Both ≈ 1.25 — the formula is exact. Memorize its
shape: **bias = (omitted coefficient) × (regression of omitted on
included)**. Two corollaries worth their weight: bias vanishes if the
omitted variable is uncorrelated with your regressor (randomization's
logic), and the *sign* of the bias is guessable from theory before any
data — "my β̂ is likely inflated by an omitted sector tilt" is a sentence
you can and should say about your own work first.

## E4 — the audit, on real data

In [ ]:
from qrc.data import get_prices
from qrc.synth import synthetic_prices
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 3.5)

if DATA_SOURCE == "real":
    rr = np.log(get_prices(["SPY", "QQQ"], start="2010-01-01")).diff().dropna()
else:
    p = synthetic_prices(n_days=3800, n_assets=2, seed=5, corr=0.85)
    rr = np.log(p).diff().dropna(); rr.columns = ["SPY", "QQQ"]

f = ols_fit(rr["SPY"].values, rr["QQQ"].values)
e = pd.Series(f["resid"], index=rr.index)

# A4: variance by time bucket
buckets = np.array_split(e, 5)
print("A4  var(e) by time quintile:", [f"{b.var():.2e}" for b in buckets])
# A5: ACF of residuals
print("A5  ACF(e) lags 1-5:", [f"{e.autocorr(l):+.3f}" for l in range(1, 6)])
# A6: normality probes
print(f"A6  excess kurtosis {stats.kurtosis(e):.1f} (0 for normal)   "
      f"skew {stats.skew(e):+.2f}")
fig, ax = plt.subplots(1, 2)
stats.probplot(e, dist="norm", plot=ax[0]); ax[0].set_title("QQ vs normal")
stats.probplot(e, dist=stats.t, sparams=(5,), plot=ax[1]); ax[1].set_title("QQ vs t(5)")
plt.tight_layout(); plt.show()

**Expected pattern.** A4: bucket variances span multiples (vol regimes —
verdict: **broken**). A5: daily ACF within ±0.05 (verdict: ~holds *here* —
day 8 manufactures the case where it cannot). A6: excess kurtosis 3–10 —
far beyond normal; the t(5) QQ straightens notably vs the normal QQ
(verdict: **broken, and mostly harmless at this n** — accept it, note it,
winsorize for stability only in day 11's sense). **The audit is five lines;
run it on every regression you ever publish.** Common mistake: ordering
residuals by size instead of time when judging A4 — homoskedasticity is a
claim about *conditional* variance; ordering by |e| manufactures any shape
you want.

## E5 — the defense (exemplar)

"Normality is assumption A6, which buys only the exact finite-sample t/F
distributions; it is not required for unbiasedness (A2), consistency, or
asymptotic validity of the t-test. The extreme value of finance is routine
non-normality: with n ≈ 2,500 daily observations the CLT controls the mean
structure, and our own size simulations (module 04) show ≤1–2% distortion
at far smaller n with t(5) parents. What we did *not* take on faith: A4 and
A5 — we tested the residuals for time-varying variance and autocorrelation
(section X) and report White/Newey–West standard errors where those
assumptions failed. The standing risks to our claims are therefore
omitted-variable bias (A2), which we address by [controls], not
non-normality of returns."

**Common mistakes across the day.** Diagnosing A2 failures with residual
plots (they are invisible there — residuals are orthogonal to included
regressors *by construction*, and the omission is not included); running
the tournament with autocorrelated errors but an iid regressor and
concluding "autocorrelation doesn't matter" (it matters precisely when the
regressor is persistent — which real signals are); believing a robust-SE
footnote can fix (d).